In [1]:
from typing import TypedDict
from typing import List, Dict, Any

from langgraph.graph import StateGraph, START, END

In [2]:
import langgraph
import langchain
import langchain_openai
import langchain_community
import langchain_tavily
import dotenv



C:\Users\Admin\AppData\Local\Temp\ipykernel_2508\3652646305.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [3]:
from dotenv import load_dotenv
import os

load_dotenv("config.env")
assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None
assert os.getenv("MISTRAL_API_KEY") is not None



In [4]:
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI

openai_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0
)

mistral_llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=os.getenv("MISTRAL_API_KEY"),
    temperature=0
)

checking whether open ai api key is working or not.If it works we use it or else we use mistral api key.

In [5]:
try:
    response = openai_llm.invoke("Say hello in one sentence.")

    print("OpenAI is working.")
    llm = openai_llm

except Exception as e:
    print("OpenAI is not working.")
    print("Switching to Mistral...")
    
    response = mistral_llm.invoke("Say good morning in one sentence.")

    print("Mistral is working.")
    llm = mistral_llm

print(response.content)

OpenAI is not working.
Switching to Mistral...


HTTPStatusError: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}

In [10]:
import os
from dotenv import load_dotenv

load_dotenv("config.env", override=True)

key = os.getenv("MISTRAL_API_KEY")

print("Key loaded:", key is not None)
print("Key length:", len(key) if key else 0)


from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=key
)

print(type(llm))
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=key
)

print(type(llm))
response = llm.invoke("Say hello in one sentence.")

print(response.content)

Key loaded: True
Key length: 32
<class 'langchain_mistralai.chat_models.ChatMistralAI'>
<class 'langchain_mistralai.chat_models.ChatMistralAI'>


HTTPStatusError: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}

Checking whether Mistral AI is working or not

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool = TavilySearchResults(
    max_results=5
)



#checking whether tavili api is working or not

In [ ]:
results = tavily_tool.invoke(
    "diabetes symptoms treatment reputable medical sources"
)

print(results)

In [ ]:
class HealthBotState(TypedDict):
    topic: str
    search_results: List[Dict[str, Any]]
    summary: str
    ready_for_quiz: bool
    quiz_question: str
    user_answer: str
    grade: str
    feedback: str
    continue_session: bool

graph = StateGraph(HealthBotState)



1. Asking the patient what health topic or medical condition they'd like to learn about.

In [ ]:
def get_topic(state: HealthBotState):
    while True:
        topic = input(
            "\nWhat health topic would you like to learn about? "
        ).strip()

        if topic:
            return {"topic": topic}

        print("Please enter a valid health topic.")

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool = TavilySearchResults(
    max_results=5
)

2. Used the Tavily search engine (via LangChain community tool) to find relevant, up-to-date medical information on the topic.

In [ ]:
def search_health_info(state: HealthBotState):
    topic = state["topic"]

    query = f"{topic} medical information site:mayoclinic.org OR site:cdc.gov OR site:nih.gov"

    results = tavily_tool.invoke(query)

    return {
        "search_results": results
    }

3. Summarized the Tavily search results into a patient-friendly explanation.

In [ ]:
def summarize_info(state: HealthBotState):
    search_results = state["search_results"]

    information = "\n\n".join(
        f"Source: {result['url']}\n{result['content']}"
        for result in search_results
    )

    prompt = f"""
You are a patient education assistant.

Using only the medical information provided , create
a clear, accurate, patient-friendly explanation about the
health topic.

Do not add medical facts that are not present in the sources.

For citations, use only the sources provided.
Do not invent, add, or cite sources that are not included
in the provided medical information.

Medical information:
{information}
"""

    response = llm.invoke(prompt)

    return {
        "summary": response.content
    }

4. Presented the summarized information to the patient and allow them to read it.

In [ ]:
def display_summary(state: HealthBotState):
    print("\n" + "=" * 60)
    print("HEALTH INFORMATION")
    print("=" * 60)
    print(state["summary"])
    print("=" * 60)

    return {}

5. Prompting the patient to indicate when they're ready for a comprehension check.

In [ ]:
def ready_for_quiz(state: HealthBotState):
    input("\nPress Enter when you are ready for the comprehension check...")

    return {
        "ready_for_quiz": True
    }

6. We Generate a single, relevant quiz question based on the summary.

In [ ]:
def generate_quiz(state: HealthBotState):
    summary = state["summary"]

    prompt = f"""
Based ONLY on the following patient education summary, create exactly ONE
comprehension question.

The question must:
- Be directly answerable from the summary.
- Test an important concept from the summary.
- Be clear and appropriate for a patient.
- Not require any outside medical knowledge.
- Contain only the question, with no answer or explanation.

Patient education summary:
{summary}
"""

    response = llm.invoke(prompt)

    return {
        "quiz_question": response.content.strip()
    }




7. Presenting the quiz question to the patient.

In [ ]:
def display_quiz(state: HealthBotState):
    print("\n" + "=" * 60)
    print("COMPREHENSION CHECK")
    print("=" * 60)
    print(state["quiz_question"])
    print("=" * 60)

    return {}

8. Allowing the patient to enter their answer to the quiz question.

In [ ]:
def get_answer(state: HealthBotState):
    answer = input("\nYour answer: ").strip()

    return {
        "user_answer": answer
    }

9. Evaluating the patient's response, providing a grade and explanation. The explanation includes relevant citations from the summary to reinforce learning.

In [ ]:
def grade_answer(state: HealthBotState):
    summary = state["summary"]
    question = state["quiz_question"]
    answer = state["user_answer"]

    prompt = f"""
You are evaluating a patient's understanding of health information.

IMPORTANT RULES:
1. Use ONLY the information contained in the health information summary.
2. Do NOT use outside medical knowledge.
3. Evaluate the patient's answer only against information that appears
   in the summary.
4. Do NOT invent facts, sources, or citations.
5. Use only source citations that are actually present in the summary.
6. Carefully compare the patient's answer with what the question asks.
7. Consider both correct and incorrect parts of the patient's answer.
8. Consider important information that the patient failed to mention.
9. Clearly explain WHY the assigned grade is appropriate.

GRADING GUIDELINES:

A = The answer is fully correct and addresses the important parts
    of the question.

B = The answer is mostly correct but contains a minor error or
    minor omission.

C = The answer demonstrates partial understanding but contains
    multiple errors or important omissions.

D = The answer demonstrates limited understanding, contains
    significant errors, or answers only a small part of the question.

F = The answer is completely incorrect, unrelated to the question,
    or provides no meaningful information.

Do not assign a grade based on information outside the summary.

HEALTH INFORMATION SUMMARY:
{summary}

QUIZ QUESTION:
{question}

PATIENT'S ANSWER:
{answer}

Evaluate the patient's answer.

Your feedback MUST contain all of the following:

1. What the patient answered correctly.
2. What the patient answered incorrectly.
3. What important information was missing, if any.
4. The correct information according to the summary.
5. A clear explanation of WHY the patient's answer deserves
   the specific grade you assigned.
6. Specific evidence from the health information summary.
7. The relevant source citation from the summary.

Return EXACTLY this format:

GRADE: <A, B, C, D, or F>

FEEDBACK:
Correct:
<Explain what the patient got right. If nothing was correct, say so.>

Incorrect:
<Explain what was wrong. If nothing was incorrect, say so.>

Missing:
<Explain important information that was missing, if any.
If nothing important was missing, say so.>

Correct information:
<State the relevant correct information using ONLY the summary.>

Why this grade:
<Explain specifically why the combination of correct, incorrect,
and missing information justifies the assigned grade.>

Evidence:
<Quote or paraphrase the relevant information from the summary.>

Source Citation:
<Give the relevant citation that appears in the summary.
Do NOT invent a citation.>
"""

    response = llm.invoke(prompt)

    result = response.content

    grade = ""
    feedback = ""

    if "GRADE:" in result:
        grade = result.split("GRADE:", 1)[1].split("FEEDBACK:", 1)[0].strip()

    if "FEEDBACK:" in result:
        feedback = result.split("FEEDBACK:", 1)[1].strip()

    return {
        "grade": grade,
        "feedback": feedback
    }

10. Presenting the grade and explanation to the patient.

In [ ]:
def display_feedback(state: HealthBotState):
    print("\n" + "=" * 60)
    print("COMPREHENSION CHECK RESULTS")
    print("=" * 60)

    print(f"\nGrade: {state['grade']}")

    print("\nFeedback:")
    print(state["feedback"])

    print("=" * 60)

    return {}

11. Asking if the patient would like to learn about another health topic or exit the session.

In [ ]:
def continue_or_exit(state: HealthBotState):
    choice = input(
        "\nWould you like to learn about another health topic? (yes/no): "
    ).strip().lower()

    return {
        "continue_session": choice == "yes"
    }

12. Either restart the flow for a new topic or end the session.

In [ ]:
def route_after_continue(state: HealthBotState):
    if state["continue_session"]:
        return "new_topic"

    return "exit"

13. Ensuring that the state is reset when starting a new topic to maintain privacy and accuracy.

In [ ]:
def reset_state(state: HealthBotState):
    return {
        "topic": "",
        "search_results": [],
        "summary": "",
        "ready_for_quiz": False,
        "quiz_question": "",
        "user_answer": "",
        "grade": "",
        "feedback": "",
        "continue_session": False
    }

In [ ]:
graph = StateGraph(HealthBotState)

In [ ]:
graph.add_node("get_topic", get_topic)
graph.add_node("search_health_info", search_health_info)
graph.add_node("summarize_info", summarize_info)
graph.add_node("display_summary", display_summary)
graph.add_node("ready_for_quiz", ready_for_quiz)
graph.add_node("generate_quiz", generate_quiz)
graph.add_node("display_quiz", display_quiz)
graph.add_node("get_answer", get_answer)
graph.add_node("grade_answer", grade_answer)
graph.add_node("display_feedback", display_feedback)
graph.add_node("continue_or_exit", continue_or_exit)
graph.add_node("reset_state", reset_state)

In [ ]:
graph.add_edge(START, "get_topic")
graph.add_edge("get_topic", "search_health_info")
graph.add_edge("search_health_info", "summarize_info")
graph.add_edge("summarize_info", "display_summary")
graph.add_edge("display_summary", "ready_for_quiz")
graph.add_edge("ready_for_quiz", "generate_quiz")
graph.add_edge("generate_quiz", "display_quiz")
graph.add_edge("display_quiz", "get_answer")
graph.add_edge("get_answer", "grade_answer")
graph.add_edge("grade_answer", "display_feedback")
graph.add_edge("display_feedback", "continue_or_exit")
graph.add_edge("reset_state", "get_topic")

In [ ]:
graph.add_conditional_edges(
    "continue_or_exit",
    route_after_continue,
    {
        "new_topic": "reset_state",
        "exit": END
    }
)

In [ ]:
app = graph.compile()

In [ ]:
result = app.invoke({
    "topic": "",
    "search_results": [],
    "summary": "",
    "ready_for_quiz": False,
    "quiz_question": "",
    "user_answer": "",
    "grade": "",
    "feedback": "",
    "continue_session": False
})

In [ ]:
import sys

print(sys.executable)
print(sys.version)